# Research Agent API - Usage Examples

This notebook demonstrates how to use the **ResearchClient** to get research responses with citations in Bigdata.com format.

## Features
- Simple synchronous interface
- Citations in standard Bigdata.com format
- Easy access to answer, citations, or both


## Setup


In [80]:
import os
import sys
import json
import logging
from IPython.display import display, Markdown, JSON

# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Configure logging for research_client module
# (basicConfig doesn't work well in Jupyter, so we configure the logger directly)
logger = logging.getLogger("research_client")
logger.setLevel(logging.INFO)

# Clear any existing handlers to avoid duplicates
logger.handlers.clear()

# Create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# Add file handler (writes to output folder)
file_handler = logging.FileHandler("output/research_client.log", mode='w')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Prevent logs from propagating to root logger (which prints to console)
logger.propagate = False

# Import the client
from research_client import ResearchClient

print("✅ Client imported successfully!")
print("✅ Logging configured (INFO level) → output/research_client.log")


✅ Client imported successfully!
✅ Logging configured (INFO level) → output/research_client.log


In [81]:
# Create client (reads BIGDATA_API_KEY from environment)
# os.environ["BIGDATA_API_KEY"] = "your-api-key-here"

client = ResearchClient()
print("✅ Client ready")


✅ Client ready


## Execute Research Query


In [82]:
# Execute research
print("🔍 Researching: 'What are the key risks Google is facing?'")
print("   This may take few seconds...\n")

query_message = """ What are the key risks Google is facing? """
#query_message = """ Generate a comprehensive daily macroeconomic morning briefing report for the US market. """
result = client.research(
    message=query_message,
    research_effort=  "lite" # "lite" OR "standard"
)

print(f"✅ Research complete!")
print(f"   Processing time: {result.processing_time_ms}ms")
print(f"   Citations found: {len(result.citations)}")


🔍 Researching: 'What are the key risks Google is facing?'
   This may take few seconds...

✅ Research complete!
   Processing time: 13022ms
   Citations found: 26


---
## A. Just Response

Display only the research answer (Markdown rendered):


In [83]:
# Get just the answer
answer = result.get_answer()

display(Markdown(answer))


Google is facing several key risks across different areas:

Firstly, the company is navigating significant challenges related to **Artificial Intelligence (AI)**. There's a concern that generative AI, by providing direct answers, could undercut Google's core advertising business and cannibalize its revenue stream, which traditionally relies on users clicking links . Additionally, integrating AI into services like Gmail carries risks if the technology malfunctions, presents misleading information, or crafts problematic emails . Privacy concerns also arise as AI delves deeper into user data to learn habits and interests . The company also faces infrastructure scaling challenges to meet the demands of AI .

Secondly, Google is under immense pressure from **regulatory and antitrust scrutiny** globally . Governments in the US, UK, and EU are investigating alleged monopolistic practices, such as pre-installing its search engine and Chrome browser on Android devices and making Google the default search engine on iPhones . There are also investigations into whether Google has imposed unfair terms on publishers and content creators, using their content for its AI services without proper compensation . Proposed remedies, like sharing online search data with rivals or the potential sale of Chrome, could introduce privacy and security risks . Antitrust trials could significantly impact Google's advertising business, a major revenue driver .

Thirdly, **security and privacy risks** are a constant concern. Google Cloud faces issues like ransomware, malware, and sophisticated hacking attempts . Google Workspace also has security vulnerabilities that could lead to data breaches, business disruptions, and financial losses . Furthermore, over two billion users are susceptible to phishing risks due to potential data leaks and the exploitation of outdated infrastructure .

Lastly, Google is exposed to **market volatility and macroeconomic risks**, which can significantly impact its stock performance . There's also the overarching risk of losing its market dominance, especially as the AI landscape rapidly evolves .

---
## B. Just Citations

Display only the citations in Bigdata.com format (JSON):


In [84]:
# Get just the citations as JSON
citations = result.get_citations()

print(f"📚 Citations ({len(citations)} sources):\n")
print(json.dumps(citations, indent=2))


📚 Citations (26 sources):

[
  {
    "id": "39024eca2e927d292cf8881547f23dff",
    "headline": "Google Stock: Valuation, Growth Drivers, and Key Risks",
    "timestamp": "2025-08-21T00:00:00",
    "source": {
      "name": "Trefis"
    },
    "url": "https://www.trefis.com/stock/goog/articles/573114/google-stock-valuation-growth-drivers-and-key-risks/2025-08-21",
    "chunks": [
      {
        "text": "Market Volatility and Macroeconomic Risks. Google's stock is highly sensitive to market downturns, often underperforming the broader S&P 500 ..."
      }
    ]
  },
  {
    "id": "560ddd180deb408c986d85578c6d486a",
    "headline": "Google Cloud Security Issues: Key Concerns",
    "timestamp": "2025-08-05T00:00:00",
    "source": {
      "name": "SentinelOne"
    },
    "url": "https://www.sentinelone.com/cybersecurity-101/cloud-security/google-cloud-security-issues/",
    "chunks": [
      {
        "text": "With ransomware, malware, and advanced hacking attempts threatening to compromi

---
## C. Response with Citations

Display both answer and citations together:


In [85]:
# Get full result as JSON (answer + citations)
full_result = result.to_dict()

print(json.dumps(full_result, indent=2))


{
  "answer": "Google is facing several key risks across different areas:\n\nFirstly, the company is navigating significant challenges related to **Artificial Intelligence (AI)**. There's a concern that generative AI, by providing direct answers, could undercut Google's core advertising business and cannibalize its revenue stream, which traditionally relies on users clicking links . Additionally, integrating AI into services like Gmail carries risks if the technology malfunctions, presents misleading information, or crafts problematic emails . Privacy concerns also arise as AI delves deeper into user data to learn habits and interests . The company also faces infrastructure scaling challenges to meet the demands of AI .\n\nSecondly, Google is under immense pressure from **regulatory and antitrust scrutiny** globally . Governments in the US, UK, and EU are investigating alleged monopolistic practices, such as pre-installing its search engine and Chrome browser on Android devices and mak

### Formatted View (Answer + Citations)


In [86]:
# Display answer as Markdown
display(Markdown("## Answer\n" + result.answer))

# Display citations in a readable format
display(Markdown("---\n## Citations"))

for i, citation in enumerate(result.citations[:10], 1):  # Show first 10
    c = citation.to_dict()
    
    # Build citation display
    parts = [f"### [{i}] {c.get('headline', 'N/A')}"]
    
    if c.get('source'):
        src = c['source']
        source_parts = []
        if src.get('name'):
            source_parts.append(f"**Source:** {src['name']}")
        if src.get('rank'):
            source_parts.append(f"**Rank:** {src['rank']}")
        if source_parts:
            parts.append(" | ".join(source_parts))
    
    if c.get('timestamp'):
        parts.append(f"**Date:** {c['timestamp']}")
    
    if c.get('url'):
        parts.append(f"**URL:** {c['url']}")
    
    # Show chunks
    if c.get('chunks'):
        parts.append("\n**Excerpts:**")
        for chunk in c['chunks']:
            text = chunk.get('text', '')
            if text:
                # Truncate long text
                display_text = text[:400] + "..." if len(text) > 400 else text
                display_text = display_text.replace('\n', ' ')
                parts.append(f"- *{display_text}*")
    
    display(Markdown("\n".join(parts) + "\n\n---"))

if len(result.citations) > 10:
    print(f"\n... and {len(result.citations) - 10} more citations")


## Answer
Google is facing several key risks across different areas:

Firstly, the company is navigating significant challenges related to **Artificial Intelligence (AI)**. There's a concern that generative AI, by providing direct answers, could undercut Google's core advertising business and cannibalize its revenue stream, which traditionally relies on users clicking links . Additionally, integrating AI into services like Gmail carries risks if the technology malfunctions, presents misleading information, or crafts problematic emails . Privacy concerns also arise as AI delves deeper into user data to learn habits and interests . The company also faces infrastructure scaling challenges to meet the demands of AI .

Secondly, Google is under immense pressure from **regulatory and antitrust scrutiny** globally . Governments in the US, UK, and EU are investigating alleged monopolistic practices, such as pre-installing its search engine and Chrome browser on Android devices and making Google the default search engine on iPhones . There are also investigations into whether Google has imposed unfair terms on publishers and content creators, using their content for its AI services without proper compensation . Proposed remedies, like sharing online search data with rivals or the potential sale of Chrome, could introduce privacy and security risks . Antitrust trials could significantly impact Google's advertising business, a major revenue driver .

Thirdly, **security and privacy risks** are a constant concern. Google Cloud faces issues like ransomware, malware, and sophisticated hacking attempts . Google Workspace also has security vulnerabilities that could lead to data breaches, business disruptions, and financial losses . Furthermore, over two billion users are susceptible to phishing risks due to potential data leaks and the exploitation of outdated infrastructure .

Lastly, Google is exposed to **market volatility and macroeconomic risks**, which can significantly impact its stock performance . There's also the overarching risk of losing its market dominance, especially as the AI landscape rapidly evolves .

---
## Citations

### [1] Google Stock: Valuation, Growth Drivers, and Key Risks
**Source:** Trefis
**Date:** 2025-08-21T00:00:00
**URL:** https://www.trefis.com/stock/goog/articles/573114/google-stock-valuation-growth-drivers-and-key-risks/2025-08-21

**Excerpts:**
- *Market Volatility and Macroeconomic Risks. Google's stock is highly sensitive to market downturns, often underperforming the broader S&P 500 ...*

---

### [2] Google Cloud Security Issues: Key Concerns
**Source:** SentinelOne
**Date:** 2025-08-05T00:00:00
**URL:** https://www.sentinelone.com/cybersecurity-101/cloud-security/google-cloud-security-issues/

**Excerpts:**
- *With ransomware, malware, and advanced hacking attempts threatening to compromise cloud environments, the risk is very high. The implementation ...*

---

### [3] Google Workspace™ Security: Top 6 Risks to Avoid in 2025
**Source:** Spin.AI
**Date:** 2024-12-03T00:00:00
**URL:** https://spin.ai/blog/google-workspace-security-top-risks/

**Excerpts:**
- *There are several Google security issues that can expose your organization to data breaches, business disruptions, financial losses, and reputational damage.*

---

...

### [7] Google's 'Cannibalization' Risk Vs Microsoft's Azure Growth: Expert Explains How AI Answers Could Slash GOOG's Ad Revenue
**Source:** Yahoo! Finance | **Rank:** RANK_2
**Date:** 2026-01-07T11:20:14
**URL:** https://finance.yahoo.com/news/googles-cannibalization-risk-vs-microsofts-110109781.html

**Excerpts:**
- *As the AI race hurtles toward 2026, market analysts are sharply divided on the fortunes of tech giants, warning that (NASDAQ:GOOG) (NASDAQ:GOOGL) Google's embrace of generative AI could severely undercut its core advertising business while favoring 's (NASDAQ:MSFT) stable cloud growth. While speaking to Schwab Network, Cory Johnson, Chief Market Strategist at Epistrophy Capital Research, argues Go...*

---

### [8] Google to face off with US government in attempt to break up company in search monopoly case
**Source:** Associated Press | **Rank:** RANK_1
**Date:** 2025-04-21T07:07:26
**URL:** https://apnews.com/article/google-search-monopoly-breakup-chrome-09b695c8de6c91b3d1e1b5305393f235

**Excerpts:**
- *Google also is sounding alarms about the proposed requirements to share online search data with rivals and the proposed sale of Chrome posing privacy and security risks. "The breadth and depth of the proposed remedies risks doing significant damage to a complex ecosystem. Some of the proposed remedies would imperil browser developers and jeopardize the digital security of millions of consumers."*

---

### [9] Google's 'Cannibalization' Risk Vs Microsoft's Azure Growth: Expert Explains How AI Answers Could Slash GOOG's Ad Revenue
**Source:** Yahoo! News | **Rank:** RANK_3
**Date:** 2026-01-07T17:24:45
**URL:** https://sg.yahoo.com/finance/news/googles-cannibalization-risk-vs-microsofts-110109781.html

**Excerpts:**
- *As the AI race hurtles toward 2026, market analysts are sharply divided on the fortunes of tech giants, warning that Alphabet Inc.-owned GOOG) GOOGL) Google's could severely undercut its core advertising business while favoring Microsoft Corp.'s MSFT) stable cloud growth. While speaking to Schwab Network, Cory Johnson, Chief Market Strategist at Epistrophy Capital Research, argues Google faces a u...*

---

### [10] Over 2B users face phishing risks after Google data leak
**Source:** Fox News | **Rank:** RANK_1
**Date:** 2025-08-26T17:53:16
**URL:** https://www.foxnews.com/tech/over-2b-users-face-phishing-risks-after-google-data-leak

**Excerpts:**
- *Google offers several 2FA methods, from SMS codes to app-based prompts and even hardware security keys. For the best protection, choose app-based or hardware verification rather than text messages. Many scams rely on exploiting outdated software. If your phone, browser, or operating system isn't up to date, attackers may use known vulnerabilities to install malware or hijack your session.*
- *Old infrastructure exploited with "dangling buckets" Separately from the Salesforce incident, Google Cloud customers are also facing another type of attack. Hackers are trying to exploit outdated access addresses using a method called the dangling bucket. This can allow them to inject malware or steal data. Both businesses and individuals are vulnerable to losing control over sensitive information...*

---


... and 16 more citations


---
## Save Results to File


In [87]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Save just citations
with open("output/citations.json", "w") as f:
    f.write(result.get_citations_json())
print("✅ Saved: output/citations.json")

# Save full result (answer + citations)
with open("output/research_result.json", "w") as f:
    f.write(result.to_json())
print("✅ Saved: output/research_result.json")


✅ Saved: output/citations.json
✅ Saved: output/research_result.json


---
## Citation Format Reference

The citations follow the standard Bigdata.com format:

```json
{
  "id": "E91DED180158906A74444B7837742178",
  "headline": "Article Title",
  "timestamp": "2026-01-06T15:00:30",
  "source": {
    "id": "5A5702",
    "name": "Benzinga",
    "rank": "RANK_1"
  },
  "url": "https://...",
  "chunks": [
    {
      "cnum": 5,
      "text": "Relevant text excerpt...",
      "relevance": 0.94,
      "sentiment": 0.82
    }
  ]
}
```

**Fields** (only non-null values are included):
- `id`: Document identifier
- `headline`: Article title
- `timestamp`: Publication date/time
- `source.id`: Source identifier
- `source.name`: Source name (e.g., "Benzinga", "Yahoo! Finance")
- `source.rank`: Source quality rank (e.g., "RANK_1")
- `url`: Document URL
- `chunks`: Array of relevant text excerpts with relevance scores


---
## D. Answer with Inline Citation Numbers

Display the answer with inline citation markers [1], [2], etc. and a numbered references section (like the screenshot):


In [88]:
# Get answer with inline citation numbers
answer_with_citations = result.get_answer_with_citations()

# Get numbered citations that match the inline numbers
numbered_citations = result.get_numbered_citations()

print(f"📊 Found {len(numbered_citations)} inline citations\n")


📊 Found 18 inline citations



In [89]:
# Display answer with inline citation numbers [1], [2], etc.
display(Markdown("## Answer\n\n" + answer_with_citations))


## Answer

Google is facing several key risks across different areas:

Firstly, the company is navigating significant challenges related to **Artificial Intelligence (AI)**. There's a concern that generative AI, by providing direct answers, could undercut Google's core advertising business and cannibalize its revenue stream, which traditionally relies on users clicking links  [18] [18] [18]. Additionally, integrating AI into services like Gmail carries risks if the technology malfunctions, presents misleading information, or crafts problematic emails  [17]. Privacy concerns also arise as AI delves deeper into user data to learn habits and interests  [17]. The company also faces infrastructure scaling challenges to meet the demands of AI  [16].

Secondly, Google is under immense pressure from **regulatory and antitrust scrutiny** globally  [15] [14] [13] [12] [10] [11]. Governments in the US, UK, and EU are investigating alleged monopolistic practices, such as pre-installing its search engine and Chrome browser on Android devices and making Google the default search engine on iPhones  [10]. There are also investigations into whether Google has imposed unfair terms on publishers and content creators, using their content for its AI services without proper compensation  [9]. Proposed remedies, like sharing online search data with rivals or the potential sale of Chrome, could introduce privacy and security risks  [8]. Antitrust trials could significantly impact Google's advertising business, a major revenue driver  [7].

Thirdly, **security and privacy risks** are a constant concern. Google Cloud faces issues like ransomware, malware, and sophisticated hacking attempts  [6]. Google Workspace also has security vulnerabilities that could lead to data breaches, business disruptions, and financial losses  [5]. Furthermore, over two billion users are susceptible to phishing risks due to potential data leaks and the exploitation of outdated infrastructure  [4].

Lastly, Google is exposed to **market volatility and macroeconomic risks**, which can significantly impact its stock performance  [3]. There's also the overarching risk of losing its market dominance, especially as the AI landscape rapidly evolves  [2] [1].

In [90]:
# Display numbered references section
display(Markdown("---\n## References\n"))

for citation in numbered_citations:
    num = citation.get('number', '?')
    headline = citation.get('headline', 'N/A')
    
    # Build citation card
    parts = [f"**[{num}]** {headline}"]
    
    # Source info
    source = citation.get('source', {})
    source_name = source.get('name') if source else citation.get('source_name')
    if source_name:
        parts.append(f"📰 **{source_name}**")
    
    # Date
    timestamp = citation.get('timestamp')
    if timestamp:
        parts.append(f"📅 {timestamp[:10]}")
    
    # URL
    url = citation.get('url')
    if url:
        parts.append(f"🔗 [{url[:50]}...]({url})")
    
    # Chunks/excerpts
    chunks = citation.get('chunks', [])
    if chunks:
        parts.append("\n**Excerpts:**")
        for chunk in chunks[:2]:  # Show max 2 excerpts
            text = chunk.get('text', '')
            if text:
                display_text = text[:300] + "..." if len(text) > 300 else text
                display_text = display_text.replace('\n', ' ')
                parts.append(f"- *{display_text}*")
    
    display(Markdown("\n".join(parts) + "\n\n---"))


---
## References


**[1]** Google's 'Cannibalization' Risk Vs Microsoft's Azure Growth: Expert Explains How AI Answers Could Slash GOOG's Ad Revenue
📰 **Benzinga**
📅 2026-01-06
🔗 [https://www.benzinga.com/node/49715103?utm_campaig...](https://www.benzinga.com/node/49715103?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

**Excerpts:**
- *As the AI race hurtles toward 2026, market analysts are sharply divided on the fortunes of tech giants, warning that Alphabet Inc.-owned (NASDAQ:GOOG) (NASDAQ:GOOGL) Google's embrace of generative AI could severely undercut its core advertising business while favoring Microsoft Corp.'s (NASDAQ:MSFT)...*

---

**[2]** FILE - Alphabet CEO Sundar Pichai speaks about Google DeepMind at a Google I/O event in Mountain View, Calif., Wednesday, May 10, 2023. (AP Photo/Jeff Chiu, File)
📰 **KSAT-TV San Antonio**
📅 2026-01-08
🔗 [https://www.ksat.com/business/2026/01/08/gmail-add...](https://www.ksat.com/business/2026/01/08/gmail-adds-new-ai-features-turning-it-into-a-personal-assistant/)

**Excerpts:**
- *But thrusting more AI into Gmail poses potential risks for Google, especially if the technology malfunctions and presents misleading information or crafts emails that get users into trouble - even though people are able to proofread the messages or turn off the features at any time.*

---

**[3]** Google's AI Infrastructure Challenge: Doubling Capacity ...
📰 **Trax Technologies**
📅 2025-12-01
🔗 [https://www.traxtech.com/ai-in-supply-chain/google...](https://www.traxtech.com/ai-in-supply-chain/googles-ai-infrastructure-challenge-doubling-capacity-every-six-months-to-meet-demand)

**Excerpts:**
- *Google faces an infrastructure-scaling challenge that defines the current AI competitive landscape: doubling its serving capacity every six ...*

---

**[4]** How Google became the internet giant at the center of a government crackdown
📰 **CNN**
📅 2025-05-09
🔗 [https://www.cnn.com/2025/05/09/business/google-int...](https://www.cnn.com/2025/05/09/business/google-internet-giant-crackdown)

**Excerpts:**
- *And it's not just Google; Meta, Microsoft, Amazon and Apple have also faced antitrust scrutiny over how they operate their tech platforms. A breakup could especially hurt Google now, as AI surges and chatbots threaten to challenge its core business. Friday marks the end of a three-week series of hea...*

---

...

**[14]** Over 2B users face phishing risks after Google data leak
📰 **Fox News**
📅 2025-08-26
🔗 [https://www.foxnews.com/tech/over-2b-users-face-ph...](https://www.foxnews.com/tech/over-2b-users-face-phishing-risks-after-google-data-leak)

**Excerpts:**
- *Google offers several 2FA methods, from SMS codes to app-based prompts and even hardware security keys. For the best protection, choose app-based or hardware verification rather than text messages. Many scams rely on exploiting outdated software. If your phone, browser, or operating system isn't up ...*
- *Old infrastructure exploited with "dangling buckets" Separately from the Salesforce incident, Google Cloud customers are also facing another type of attack. Hackers are trying to exploit outdated access addresses using a method called the dangling bucket. This can allow them to inject malware or ste...*

---

**[15]** Google Workspace™ Security: Top 6 Risks to Avoid in 2025
📰 **Spin.AI**
📅 2024-12-03
🔗 [https://spin.ai/blog/google-workspace-security-top...](https://spin.ai/blog/google-workspace-security-top-risks/)

**Excerpts:**
- *There are several Google security issues that can expose your organization to data breaches, business disruptions, financial losses, and reputational damage.*

---

**[16]** Google Stock: Valuation, Growth Drivers, and Key Risks
📰 **Trefis**
📅 2025-08-21
🔗 [https://www.trefis.com/stock/goog/articles/573114/...](https://www.trefis.com/stock/goog/articles/573114/google-stock-valuation-growth-drivers-and-key-risks/2025-08-21)

**Excerpts:**
- *Market Volatility and Macroeconomic Risks. Google's stock is highly sensitive to market downturns, often underperforming the broader S&P 500 ...*

---

**[17]** Google was at risk of losing its dominance
📰 **CNBC**
📅 2025-12-20
🔗 [https://www.cnbc.com/2025/12/20/josh-woodward-goog...](https://www.cnbc.com/2025/12/20/josh-woodward-google-gemini-ai-safety.html)

**Excerpts:**
- *Alphabet shares plunged 18% in the first quarter, their worst performance for any period since 2022, and concerns were building that the company ...*

---

**[18]** Google's Future: AI Challenges & Opportunities
📰 **Medium · Sorin Ciornei**
🔗 [https://medium.com/thereach-ai/googles-future-chal...](https://medium.com/thereach-ai/googles-future-challenges-and-opportunities-in-an-ai-driven-world-519a673868a9)

**Excerpts:**
- *Google has two big challenges that is yet to solve. Google, a titan in the tech industry, or for some, the ad industry, may be facing an existential threat.*

---

### JSON Export with Inline Citations


In [91]:
# Export as JSON with inline citations in the answer
result_with_inline = result.to_dict_with_inline_citations()

print(json.dumps(result_with_inline, indent=2)[:3000] + "\n... (truncated)")


{
  "answer": "Google is facing several key risks across different areas:\n\nFirstly, the company is navigating significant challenges related to **Artificial Intelligence (AI)**. There's a concern that generative AI, by providing direct answers, could undercut Google's core advertising business and cannibalize its revenue stream, which traditionally relies on users clicking links  [18] [18] [18]. Additionally, integrating AI into services like Gmail carries risks if the technology malfunctions, presents misleading information, or crafts problematic emails  [17]. Privacy concerns also arise as AI delves deeper into user data to learn habits and interests  [17]. The company also faces infrastructure scaling challenges to meet the demands of AI  [16].\n\nSecondly, Google is under immense pressure from **regulatory and antitrust scrutiny** globally  [15] [14] [13] [12] [10] [11]. Governments in the US, UK, and EU are investigating alleged monopolistic practices, such as pre-installing its

In [92]:
# Save result with inline citations
with open("output/result_with_inline_citations.json", "w") as f:
    f.write(result.to_json_with_inline_citations())
print("✅ Saved: output/result_with_inline_citations.json")


✅ Saved: output/result_with_inline_citations.json
